# Phase 7: Classification Threshold Analysis for Logistic Regression

**Project:** Customer Churn Prediction & Retention Analytics  
**Model:** Preferred Balanced Logistic Regression (`models/logistic_regression_pipeline.joblib`)  
**Evaluation Set:** Identical 20% stratified test set ($N = 1,409$, 374 actual churners)  

### Objectives:
1. Extract continuous predicted probabilities of customer churn ($P(\text{Churn} = 1)$) on the test set.
2. Evaluate decision thresholds from **0.30 to 0.70** in steps of 0.05.
3. Track trade-offs across Accuracy, Precision, Recall, F1-Score, and False Positives vs. False Negatives.
4. Save comprehensive threshold metrics to `outputs/logistic_threshold_analysis.csv`.
5. Plot diagnostic curves: Precision-Recall-F1 Trade-off and Classification Error Progression.
6. Identify key benchmark thresholds (Highest F1, Highest Recall, Highest Precision, Default 0.50) and establish a defensible recommendation framework.

## 1. Import Libraries

In [ ]:
import sys
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Set visualization styles
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["axes.edgecolor"] = "#cccccc"

sys.path.append("..")
from src.preprocess import get_train_test_data

fig_dir = Path("../outputs/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Pipeline & Extract Predicted Probabilities

In [ ]:
data = get_train_test_data(file_path="../data/processed/telco_churn_clean.csv")
X_test, y_test = data["X_test"], data["y_test"]

pipeline = joblib.load("../models/logistic_regression_pipeline.joblib")
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(f"Generated predicted churn probabilities for {len(y_prob):,} test customers.")
print(f"Probability range: Min = {y_prob.min():.4f}, Max = {y_prob.max():.4f}, Mean = {y_prob.mean():.4f}")

## 3. Threshold Sweep (0.30 - 0.70)

In [ ]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
rows = []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    predicted_churn = int(tp + fp)
    
    rows.append({
        "Threshold": t,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "True_Negatives": int(tn),
        "False_Positives": int(fp),
        "False_Negatives": int(fn),
        "True_Positives": int(tp),
        "Predicted_Churn_Risk": predicted_churn,
        "Detection_Rate_Pct": round(rec * 100, 2),
        "Miss_Rate_Pct": round((fn / (tp + fn)) * 100, 2)
    })

df_res = pd.DataFrame(rows)
df_res.to_csv(output_dir / "logistic_threshold_analysis.csv", index=False)
display(df_res)

## 4. Precision, Recall & F1-Score Trade-off Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(df_res["Threshold"], df_res["Precision"], marker="o", lw=2.2, color="#e67e22", label="Precision (Class 1)")
ax.plot(df_res["Threshold"], df_res["Recall"], marker="s", lw=2.2, color="#2980b9", label="Recall (Class 1)")
ax.plot(df_res["Threshold"], df_res["F1_Score"], marker="^", lw=2.2, color="#27ae60", label="F1-Score (Class 1)")
ax.axvline(0.50, color="#7f8c8d", linestyle="--", lw=1.5, label="Default Threshold (0.50)")
ax.axvline(0.60, color="#8e44ad", linestyle=":", lw=1.5, label="Peak F1 Threshold (0.60)")

ax.set_title("Logistic Regression: Precision, Recall & F1 vs. Classification Threshold", fontsize=13, pad=12, weight="bold")
ax.set_xlabel("Classification Decision Threshold", fontsize=12)
ax.set_ylabel("Metric Score", fontsize=12)
ax.set_ylim(0.35, 1.0)
ax.set_xticks(thresholds)
ax.legend(loc="lower left", fontsize=10.5)
plt.tight_layout()
plt.savefig(fig_dir / "logistic_threshold_tradeoff.png", dpi=300)
plt.show()

## 5. Classification Error Progression (False Positives vs. False Negatives)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(df_res["Threshold"], df_res["False_Positives"], marker="o", lw=2.2, color="#e74c3c", label="False Positives (False Alarms)")
ax.plot(df_res["Threshold"], df_res["False_Negatives"], marker="s", lw=2.2, color="#9b59b6", label="False Negatives (Missed Churners)")
ax.axvline(0.50, color="#7f8c8d", linestyle="--", lw=1.5, label="Default Threshold (0.50)")
ax.axvline(0.60, color="#8e44ad", linestyle=":", lw=1.5, label="Peak F1 Threshold (0.60)")

ax.set_title("Logistic Regression: Classification Errors vs. Decision Threshold", fontsize=13, pad=12, weight="bold")
ax.set_xlabel("Classification Decision Threshold", fontsize=12)
ax.set_ylabel("Error Count (Instances)", fontsize=12)
ax.set_xticks(thresholds)
ax.legend(loc="upper center", fontsize=10.5)
plt.tight_layout()
plt.savefig(fig_dir / "logistic_threshold_errors.png", dpi=300)
plt.show()

## 6. Key Benchmark Thresholds & Trade-off Analysis

### Benchmark Summary:
1. **Peak F1-Score Benchmark (Threshold = 0.60):**
   - **Precision:** 55.95% | **Recall:** 71.66% | **F1-Score:** **0.6284 (Peak)** | **Accuracy:** 77.50%
   - **Confusion:** TP = 268, FP = 211, FN = 106, TN = 824
   - *Trade-off:* Compared to the default 0.50 threshold, moving to 0.60 reduces False Positives from 280 to 211 (69 fewer false alarms), but increases False Negatives from 82 to 106 (24 additional missed churners).

2. **Default Baseline (Threshold = 0.50):**
   - **Precision:** 51.05% | **Recall:** **78.07%** | **F1-Score:** 0.6173 | **Accuracy:** 74.31%
   - **Confusion:** TP = 292, FP = 280, FN = 82, TN = 755
   - *Trade-off:* Captures almost 4 out of 5 churners (292 / 374) while maintaining precision above 50%.

3. **Maximum Recall Benchmark (Threshold = 0.30):**
   - **Precision:** 43.69% | **Recall:** **90.64%** | **F1-Score:** 0.5896 | **Accuracy:** 66.50%
   - **Confusion:** TP = 339, FP = 437, FN = 35, TN = 598
   - *Trade-off:* Catches >90% of all churners (misses only 35), but flags 776 customers (more than half the test set) with 437 false alarms.

4. **Maximum Precision Benchmark (Threshold = 0.70):**
   - **Precision:** **60.42%** | **Recall:** 62.03% | **F1-Score:** 0.6121 | **Accuracy:** 79.13%
   - **Confusion:** TP = 232, FP = 152, FN = 142, TN = 883
   - *Trade-off:* Minimizes false alarms to 152, but misses 37.97% of all churners (142 missed).

### Statistical vs. Business Decision Framework:
- **Statistical Recommendation:** Empirically, **0.60** represents the optimal point of harmonic balance ($F1 = 0.6284$), while **0.50** provides superior recall ($78.07%$) for proactive retention.
- **Business Threshold Requirement:** Setting a definitive production threshold requires specific operational unit economics: the exact dollar cost of a retention action (e.g., outreach email vs. retention discount) versus the expected lifetime value ($CLTV$) lost per unprevented churner.